# META Stock Price Prediction Project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv("META.csv")
print(df.head())

print(df.info())

print(df.isnull().sum())

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df.reset_index(drop=True, inplace=True)

plt.figure(figsize=(14,6))
plt.plot(df['Date'], df['Close'], color='blue')
plt.title('META Closing Price Trend')
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.grid(True)
plt.show()

plt.figure(figsize=(14,6))
plt.plot(df['Date'], df['Volume'], color='green')
plt.title('META Trading Volume')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.grid(True)
plt.show()

plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True),
            annot=True,
            cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

df['Prev_Close'] = df['Close'].shift(1)
df['Prev_Open'] = df['Open'].shift(1)
df['Prev_High'] = df['High'].shift(1)
df['Prev_Low'] = df['Low'].shift(1)
df['Prev_Volume'] = df['Volume'].shift(1)

df.dropna(inplace=True)

print(df.head())

X = df[['Prev_Close',
        'Prev_Open',
        'Prev_High',
        'Prev_Low',
        'Prev_Volume']]

y = df['Close']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Training Completed")

y_pred = model.predict(X_test)

prediction_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})

print(prediction_df.head())

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", mae)
print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("R2 Score:", r2)

plt.figure(figsize=(14,6))
plt.plot(y_test.values, label='Actual Price')
plt.plot(y_pred, label='Predicted Price')
plt.title("Actual vs Predicted Stock Prices")
plt.xlabel("Days")
plt.ylabel("Stock Price")
plt.legend()
plt.show()

importance = model.feature_importances_
feature_names = X.columns

feature_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
})

feature_df = feature_df.sort_values(
    by='Importance',
    ascending=False
)

print(feature_df)

plt.figure(figsize=(10,5))
plt.bar(feature_df['Feature'],
        feature_df['Importance'])

plt.title("Feature Importance")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.show()

last_row = X.iloc[-1].values.reshape(1, -1)

future_price = model.predict(last_row)

print("Predicted Next Day Closing Price:", future_price[0])

future_days = 30
future_predictions = []

current_input = X.iloc[-1].values

for i in range(future_days):
    pred = model.predict([current_input])[0]
    future_predictions.append(pred)
    current_input[0] = pred

plt.figure(figsize=(12,6))
plt.plot(range(1, future_days+1),
         future_predictions,
         marker='o')

plt.title("Future Stock Price Trend")
plt.xlabel("Future Days")
plt.ylabel("Predicted Price")
plt.grid(True)
plt.show()
